# Tamamen Lokal RAG — 50 Satırda

> **Hafta 13 · Lokal RAG ve Kendi Verinle Konuşma** · *Üretken YZ Atölyesi · Dr. Murat Altun*

Hiçbir bulut API kullanmadan. Ollama + ChromaDB + Python.

---


## 📦 Kurulum

Bu notebook'taki kodu çalıştırmak için aşağıdaki paketler gerekir:


In [ ]:
# pip install ollama chromadb pypdf


## PDF'leri Oku




In [ ]:
from pypdf import PdfReader

import os



def pdf_text(yol):

    pdf = PdfReader(yol)

    return "\n".join(p.extract_text() for p in pdf.pages if p.extract_text())



pdf_klasoru = "./belgelerim"  # PDF'lerini buraya koy

metinler = {}

for f in os.listdir(pdf_klasoru):

    if f.endswith(".pdf"):

        metinler[f] = pdf_text(os.path.join(pdf_klasoru, f))

print(f"{len(metinler)} PDF okundu")



## Chunk'la




In [ ]:
def chunkla(metin, boyut=500, overlap=50):

    chunks = []

    for i in range(0, len(metin), boyut - overlap):

        chunks.append(metin[i:i + boyut])

    return chunks



tum_chunks = []

for dosya, metin in metinler.items():

    for i, c in enumerate(chunkla(metin)):

        tum_chunks.append({"id": f"{dosya}_{i}", "text": c, "kaynak": dosya})

print(f"{len(tum_chunks)} chunk üretildi")



## ChromaDB'ye Yaz (lokal embedding)




In [ ]:
import chromadb

from chromadb.utils.embedding_functions import OllamaEmbeddingFunction



embed_fn = OllamaEmbeddingFunction(

    url="http://localhost:11434/api/embeddings",

    model_name="nomic-embed-text",

)

client = chromadb.PersistentClient(path="./chroma_db")

col = client.get_or_create_collection("benim_kitap", embedding_function=embed_fn)



col.add(

    documents=[c["text"] for c in tum_chunks],

    metadatas=[{"kaynak": c["kaynak"]} for c in tum_chunks],

    ids=[c["id"] for c in tum_chunks],

)

print("İndekslendi")



## Sor




In [ ]:
import ollama



def sor(soru, k=4):

    sonuc = col.query(query_texts=[soru], n_results=k)

    baglam = "\n\n".join(sonuc["documents"][0])

    kaynaklar = list(set(m["kaynak"] for m in sonuc["metadatas"][0]))



    cevap = ollama.chat(

        model="llama3.2:3b",

        messages=[

            {"role": "system", "content": "Türkçe cevap ver. Sadece verilen bağlamdan yararlan, kaynak göster."},

            {"role": "user", "content": f"Bağlam:\n{baglam}\n\nSoru: {soru}\n\nCevap:"}

        ]

    )

    return cevap["message"]["content"], kaynaklar



cevap, kaynaklar = sor("Bu kitapların ana teması nedir?")

print(cevap)

print("\nKaynaklar:", kaynaklar)



---
## 🛠️ Sen Yap

**Görev 1.** Bu kodu kendi PDF'lerinle çalıştır
**Görev 2.** 3 farklı chunk_size (300/500/1000) dene, en iyiyi seç


---
*— Üretken YZ Atölyesi · Dr. Murat Altun · 2026*
